<a href="https://colab.research.google.com/github/arses-ui/Barnacles/blob/main/Trained_model/YOLOv8kfoldtraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path

dataset_path = Path("/content/drive/MyDrive/yolo_dataset")
labels = sorted(dataset_path.rglob("*labels/*.txt"))  #
print(labels)

[PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0000.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0001.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0002.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0003.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0004.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0005.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0006.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0007.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0008.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0009.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0010.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labels/image1_patch_0011.txt'), PosixPath('/content/drive/MyDrive/yolo_dataset/labe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
classes =["barnacle"]
cls_idx = [0]

In [ ]:
import pandas as pd

index = [label.stem for label in labels]
labels_df = pd.DataFrame([], columns =cls_idx, index=index)

In [ ]:
from collections import Counter

for label in labels:
    lbl_counter = Counter()

    with open(label) as lf:
        lines = lf.readlines()

    for line in lines:
        # classes for YOLO label uses integer at first position of each line
        lbl_counter[int(line.split(" ", 1)[0])] += 1

    labels_df.loc[label.stem] = lbl_counter

labels_df = labels_df.fillna(0.0)  # replace `nan` values with `0.0`

/tmp/ipython-input-21-1735700115.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labels_df = labels_df.fillna(0.0)  # replace `nan` values with `0.0`


In [ ]:
print(labels_df)

                    0
image1_patch_0000  50
image1_patch_0001  59
image1_patch_0002  78
image1_patch_0003  79
image1_patch_0004  47
...                ..
image2_patch_0057   8
image2_patch_0058   7
image2_patch_0061   9
image2_patch_0062  18
image2_patch_0063  26

[124 rows x 1 columns]


In [ ]:
!pip install  scikit-learn

In [ ]:
import random
from sklearn.model_selection import KFold
random.seed(0)
ksplit= 5
kf =KFold(n_splits=ksplit, shuffle=True, random_state=20)
kfolds = list(kf.split(labels_df))

In [ ]:
folds = [f"split_{n}" for n in range(1, ksplit + 1)]
folds_df = pd.DataFrame(index=index, columns=folds)

for i, (train, val) in enumerate(kfolds, start=1):
  folds_df.loc[labels_df.iloc[train].index, f"split_{i}"] = "train"
  folds_df.loc[labels_df.iloc[val].index, f"split_{i}"] = "val"

In [ ]:
fold_lbl_distrb = pd.DataFrame(index=folds, columns=cls_idx)

for n, (train_indices, val_indices) in enumerate(kfolds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()

    # To avoid division by zero, we add a small value (1E-7) to the denominator
    ratio = val_totals / (train_totals + 1e-7)
    fold_lbl_distrb.loc[f"split_{n}"] = ratio

In [ ]:
import datetime
import yaml

supported_extensions = [".jpg", ".jpeg", ".png"]

# Initialize an empty list to store image file paths
images = []

# Loop through supported extensions and gather image files
for ext in supported_extensions:
    images.extend(sorted((dataset_path / "images").rglob(f"*{ext}")))

# Create the necessary directories and dataset YAML files
save_path = Path(dataset_path / f"{datetime.date.today().isoformat()}_{ksplit}-Fold_Cross-val")
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    # Create directories
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "labels").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "labels").mkdir(parents=True, exist_ok=True)

    # Create dataset YAML files
    dataset_yaml = split_dir / f"{split}_dataset.yaml"
    ds_yamls.append(dataset_yaml)

    with open(dataset_yaml, "w") as ds_y:
        yaml.safe_dump(
            {
                "path": split_dir.as_posix(),
                "train": "train",
                "val": "val",
                "names": classes,
            },
            ds_y,
        )

In [ ]:
import shutil

from tqdm import tqdm

for image, label in tqdm(zip(images, labels), total=len(images), desc="Copying files"):
    for split, k_split in folds_df.loc[image.stem].items():
        # Destination directory
        img_to_path = save_path / split / k_split / "images"
        lbl_to_path = save_path / split / k_split / "labels"

        # Copy image and label files to new directory (SamefileError if file already exists)
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

Copying files: 100%|██████████| 124/124 [00:16<00:00,  7.35it/s]


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 49.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
from ultralytics import YOLO

weights_path = "yolov8s-seg.pt"  # using small model(11.8 million parameters)
model = YOLO(weights_path, task="segment") #specifying to train on image segmentation rather than just detection

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 22.8M/22.8M [00:00<00:00, 99.5MB/s]


In [ ]:
results = {}

# Define your additional arguments here
batch = 16
project = "Kfold_training_barnacles"
epochs = 20

for k, dataset_yaml in enumerate(ds_yamls):
    model = YOLO(weights_path, task="detect")
    results[k] = model.train(
        data=dataset_yaml, epochs=epochs, batch=batch, project=project, name=f"fold_{k + 1}"
    )  # include any additional train arguments

Ultralytics 8.3.170 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_dataset/2025-07-27_5-Fold_Cross-val/split_1/split_1_dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fold_12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

train: Scanning /content/drive/MyDrive/yolo_dataset/2025-07-27_5-Fold_Cross-val/split_1/train/labels.cache... 99 images, 0 backgrounds, 0 corrupt: 100%|██████████| 99/99 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 1.0±0.5 ms, read: 38.4±17.9 MB/s, size: 110.4 KB)


val: Scanning /content/drive/MyDrive/yolo_dataset/2025-07-27_5-Fold_Cross-val/split_1/val/labels.cache... 25 images, 0 backgrounds, 0 corrupt: 100%|██████████| 25/25 [00:00<?, ?it/s]


...


20 epochs completed in 0.050 hours.
Optimizer stripped from Kfold_training_barnacles/fold_5/weights/last.pt, 23.8MB
Optimizer stripped from Kfold_training_barnacles/fold_5/weights/best.pt, 23.8MB

Validating Kfold_training_barnacles/fold_5/weights/best.pt...
Ultralytics 8.3.170 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 42.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95):   0%|          | 0/1 [00:00<?, ?it/s]

WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]


                   all         24        974      0.932      0.836      0.903      0.569      0.667       0.59      0.488     0.0947
Speed: 0.2ms preprocess, 7.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to Kfold_training_barnacles/fold_5


In [ ]:
!zip -r KFold_training_barnacles.zip Kfold_training_barnacles

  adding: Kfold_training_barnacles/ (stored 0%)
  adding: Kfold_training_barnacles/fold_4/ (stored 0%)
  adding: Kfold_training_barnacles/fold_4/args.yaml (deflated 52%)
  adding: Kfold_training_barnacles/fold_4/MaskPR_curve.png (deflated 22%)
  adding: Kfold_training_barnacles/fold_4/BoxPR_curve.png (deflated 23%)
  adding: Kfold_training_barnacles/fold_4/train_batch1.jpg (deflated 1%)
  adding: Kfold_training_barnacles/fold_4/confusion_matrix_normalized.png (deflated 36%)
  adding: Kfold_training_barnacles/fold_4/train_batch71.jpg (deflated 4%)
  adding: Kfold_training_barnacles/fold_4/train_batch70.jpg (deflated 2%)
  adding: Kfold_training_barnacles/fold_4/labels_correlogram.jpg (deflated 38%)
  adding: Kfold_training_barnacles/fold_4/train_batch2.jpg (deflated 1%)
  adding: Kfold_training_barnacles/fold_4/labels.jpg (deflated 31%)
  adding: Kfold_training_barnacles/fold_4/MaskF1_curve.png (deflated 17%)
  adding: Kfold_training_barnacles/fold_4/BoxR_curve.png (deflated 16%)
  addi